# Linke turbidity — project analysis and presentation

Research-style walkthrough of the OLUOL station analysis, methodology, monthly comparison, irradiance implications, and the selected June 2022 aerosol case study.

**Data requirement:** place the authorised `OLUOL.cleaned.nc` file in `data/`. See `data/README.md`. Some figures are generated from the analysis cells; the former local MODIS screenshot is not bundled.

**Interpretation:** numerical findings are exploratory and should be treated as preliminary until the underlying calculations, filtering choices, and comparisons have been independently rechecked. Outputs have been cleared; execute cells from top to bottom.


# Validating Climatological Linke Turbidity at Station OLUOL

**Eye2Sky Network (DLR) — Oldenburg, Germany**




# Project Description

**Objective:** validate the climatological Linke turbidity (T_L) commonly used in clear-sky
solar irradiance models against T_L computed directly from our own station's measured
direct normal irradiance (DNI), following the airmass-independent formulation of
**Ineichen & Perez (2002)**.

### Why turbidity matters
Linke turbidity is the single parameter in the Ineichen clear-sky model that encodes how much
the atmosphere (aerosols + water vapor) attenuates solar radiation. It directly determines
predicted GHI/DNI/DHI — getting it wrong biases solar resource assessments and PV yield
forecasts.

### Research questions
1. How accurate is the climatological T_L database for our specific location?
2. Does climatology systematically **over-** or **under-estimate** turbidity?
3. Are there **seasonal patterns** in that accuracy?
4. Can we identify **extreme deviations**, and explain them?

**Reference:** Ineichen, P., Perez, R. (2002). *A new airmass independent formulation for the
Linke turbidity coefficient.* Solar Energy, 73(3), 151–157.


# Data

| | |
|---|---|
| **Station** | OLUOL — Oldenburg, Germany (Eye2Sky network, operated by DLR) |
| **Location** | 53.153°N, 8.162°E, 21 m a.s.l. |
| **Period** | 2022-04-01 to 2023-04-01 |
| **Resolution** | 60 s |
| **Quality level** | Quality-controlled + cleaned (`OLUOL.cleaned.nc`) |

### Key variables used

| Variable | Description |
|---|---|
| `GHI`, `DNI`, `DHI` | Measured global / direct-normal / diffuse horizontal irradiance |
| `cGHI`, `cDNI`, `cDHI` | Modelled clear-sky irradiance (Ineichen-Perez) |
| `SZA`, `SAZ` | Solar zenith / azimuth angle (pre-computed) |
| `turbidity` | Climatological Linke turbidity used by the file's own clear-sky model |



In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pvlib

ds = xr.open_dataset("data/OLUOL.cleaned.nc")
altitude = ds.attrs["elevation"]
latitude, longitude = float(ds["latitude"].values), float(ds["longitude"].values)
ds


# Methodology


### 1. Clear-sky detection — Reno & Hansen (2016)
A sliding-window algorithm comparing measured GHI against modelled clear-sky GHI (mean level,
variability, and slope must all match) — implemented in `pvlib.clearsky.detect_clearsky`.

*Reference: Reno, M.J., Hansen, C.W. (2016). Identification of periods of clear sky irradiance
in time series of GHI measurements. Renewable Energy, 90, 520–531.*

### 2. Airmass 
Pressure/altitude-corrected ("absolute") airmass as stated in  (Ineichen & Perez, 2002)

### 3. Linke turbidity retrieval (Ineichen & Perez, 2002)

$$T_L = 1 + \dfrac{\ln\left(\dfrac{b \cdot I_0}{DNI_{measured}}\right)}{0.09 \cdot AM}
\qquad b = 0.664 + \dfrac{0.163}{f_{h1}}, \quad f_{h1}=e^{-altitude/8000}$$




| $T_L$ | Linke turbidity (the value we're solving for) |

| $DNI_{measured}$ | Measured direct normal irradiance (from our station) |

| $I_0$ | Extraterrestrial normal irradiance, corrected for Earth-Sun distance |

| $AM$ | Airmass, altitude/pressure-corrected (absolute) |

| $b$ | Altitude-dependent beam coefficient |

| $f_{h1}$ | Altitude correction factor (accounts for station elevation above sea level) |

| $altitude$ | Station elevation above sea level (21 m for OLUOL) |

**Airmass independence:** the formula is constructed so that, for a truly constant
atmosphere, the retrieved $T_L$ comes out the same regardless of which airmass the
measurement was taken at — this is the property "airmass independent" refers to, not
the absence of $AM$ from the equation itself.

Applied only at clear-sky, airmass-filtered timestamps.

### 4.Implementation
`pvlib.clearsky.lookup_linke_turbidity` — the standard gridded climatology (Rigollier et al.,
2000; Ineichen, 2008) bundled with pvlib.


In [ ]:
daytime_mask = (ds["SZA"] < 75).values
ghi_series, cghi_series = ds["GHI"].to_series(), ds["cGHI"].to_series()

clear_sky_mask = pvlib.clearsky.detect_clearsky(
    measured=ghi_series, clearsky=cghi_series, times=ghi_series.index
).values & daytime_mask

airmass_rel = pvlib.atmosphere.get_relative_airmass(ds["SZA"].to_series(), model="kastenyoung1989")
pressure = pvlib.atmosphere.alt2pres(altitude)
ds["airmass"] = ("time", pvlib.atmosphere.get_absolute_airmass(airmass_rel, pressure).values)

final_mask = clear_sky_mask
print(f"Clear-sky points: {final_mask.sum()} / {len(final_mask)}")

In [ ]:
doy = ds["time"].dt.dayofyear.values
I0 = 1367 * (1 + 0.033 * np.cos(2 * np.pi * doy / 365))
fh1 = np.exp(-altitude / 8000.0)
b = 0.664 + 0.163 / fh1

TL_derived = np.full(len(ds["DNI"]), np.nan)
valid = final_mask & (ds["DNI"].values > 0)
TL_derived[valid] = 1 + np.log(b * I0[valid] / ds["DNI"].values[valid]) / (0.09 * ds["airmass"].values[valid])
ds["TL_derived"] = ("time", TL_derived)

ds["TL_climatology"] = ("time", pvlib.clearsky.lookup_linke_turbidity(
    ds["time"].to_index(), latitude, longitude).values)


# Analysis

## Main finding: climatology systematically overestimates turbidity

Comparing monthly **medians** (self-computed, robust to outliers) against the climatological
**mean** for each month:


In [ ]:
import matplotlib.dates as mdates

valid = ds["final_mask"].values if "final_mask" in ds else ~np.isnan(ds["TL_derived"].values)

fig, ax = plt.subplots(figsize=(11,4.5))
ax.scatter(ds["time"].values[valid], ds["TL_derived"].values[valid], s=6, alpha=0.5,
           color="tab:blue", label="Self-computed T$_L$ (clear-sky, filtered)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax.set_ylabel("Linke turbidity T$_L$")
ax.set_title("Self-Computed Clear-Sky Linke Turbidity — OLUOL, Apr 2022 to Apr 2023")
ax.legend()
plt.tight_layout()
plt.show()
df = ds[["TL_derived"]].to_dataframe().dropna()
df = ds[["TL_derived"]].to_dataframe().dropna()
print(df["TL_derived"].idxmax(), df["TL_derived"].max())



Every point above is a single clear-sky, airmass-filtered timestamp — the raw material
behind the monthly comparison that follows. The visible seasonal envelope (lower in winter,
higher and more scattered in summer) foreshadows the monthly pattern shown next, and the
sparse/empty stretches (e.g. December) mark periods with too little clear-sky data to draw
conclusions from.


In [ ]:
derived = ds["TL_derived"].to_series().dropna()
clim = ds["TL_climatology"].to_series()

comparison = pd.DataFrame({
    "TL_derived_median": derived.groupby(derived.index.to_period("M")).median(),
    "n_clear_points": derived.groupby(derived.index.to_period("M")).count(),
    "TL_climatology": clim.groupby(clim.index.to_period("M")).mean()
})
comparison["overestimation_%"] = 100 * (comparison["TL_climatology"] - comparison["TL_derived_median"]) / comparison["TL_derived_median"]
comparison.index = comparison.index.to_timestamp()
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(comparison.index, comparison["TL_derived_median"], marker="o", label="Self-computed (monthly median)")
ax.plot(comparison.index, comparison["TL_climatology"], marker="o", label="pvlib climatology")
ax.set_ylabel("Linke turbidity T$_L$")
ax.set_title("Self-Computed vs Climatological T$_L$ — OLUOL, Apr 2022 to Apr 2023")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Figure 2 — Monthly relative deviation
from pathlib import Path
Path("figures").mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    comparison.index.strftime("%b %Y"),
    comparison["overestimation_%"],
    edgecolor="black",
    linewidth=0.6
)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_ylabel("Relative difference (%)")
ax.set_xlabel("Month")
ax.set_title("Monthly Difference: Climatological vs Station-Derived Linke Turbidity")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
for bar, value in zip(bars, comparison["overestimation_%"]):
    ax.annotate(f"{value:.1f}%",
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig("figures/fig02_monthly_relative_deviation.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/fig02_monthly_relative_deviation.png")

In [ ]:
# Figure 3 — Monthly climatology vs station-derived scatter comparison
x = comparison["TL_derived_median"]
y = comparison["TL_climatology"]
valid = x.notna() & y.notna()
xv, yv = x[valid], y[valid]

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(xv, yv, s=55, alpha=0.85, edgecolor="black", linewidth=0.5)

limits = [
    min(float(xv.min()), float(yv.min())),
    max(float(xv.max()), float(yv.max()))
]
ax.plot(limits, limits, linestyle="--", color="black", label="1:1 line")
for date, xval, yval in zip(xv.index, xv, yv):
    ax.annotate(date.strftime("%b"), (xval, yval),
                xytext=(4, 4), textcoords="offset points", fontsize=8)

ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Station-derived monthly median $T_L$")
ax.set_ylabel("Climatological monthly mean $T_L$")
ax.set_title("Monthly Climatology vs Station-Derived Linke Turbidity")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig("figures/fig03_monthly_turbidity_scatter.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/fig03_monthly_turbidity_scatter.png")

**Answers to research questions 1–3:**

- **Accuracy:** climatology deviates from station measurements by **15–41%** depending on month
- **Direction:** climatology **overestimates** turbidity in *every single month* of the year —
  a consistent, one-directional bias, not random scatter
- **Seasonality:** the gap is smallest in July (~15%) and largest in August (~41%) — no simple
  single-season rule, but consistently sizeable

*Caveat: self-computed T_L is derived only from the cleanest clear-sky moments (strict
filtering), while climatology represents an all-conditions long-term average — part of the
gap may reflect this selection effect rather than a pure climatology error.*




## What does this overestimation mean for predicted irradiance?

Feeding climatology's T_L vs. our measured T_L into the same Ineichen clear-sky model, for the
month with the **lowest** and **highest** climatological turbidity, isolates the practical
effect of the bias — same location, same day, same sun path, only T_L differs.


In [ ]:
from pvlib import solarposition, irradiance, clearsky

lowest_month = comparison["TL_climatology"].idxmin().to_period("M")
highest_month = comparison["TL_climatology"].idxmax().to_period("M")

def clearsky_for_month(period, day_of_month=15):
    day = pd.Timestamp(period.year, period.month, day_of_month)
    times = pd.date_range(day, day + pd.Timedelta(hours=23, minutes=59), freq="1min", tz="UTC")
    apparent_zenith = solarposition.get_solarposition(times, latitude, longitude)["apparent_zenith"]
    am_abs = pvlib.atmosphere.get_absolute_airmass(
        pvlib.atmosphere.get_relative_airmass(apparent_zenith), pressure)
    dni_extra = irradiance.get_extra_radiation(times)

    row = comparison.loc[period.to_timestamp()]
    cs_clim = clearsky.ineichen(apparent_zenith, am_abs, row["TL_climatology"], altitude, dni_extra)
    cs_meas = clearsky.ineichen(apparent_zenith, am_abs, row["TL_derived_median"], altitude, dni_extra)
    return times, cs_clim, cs_meas, row["TL_climatology"], row["TL_derived_median"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for col, period in enumerate([lowest_month, highest_month]):
    times, cs_clim, cs_meas, tl_c, tl_m = clearsky_for_month(period)
    for row_i, (cs, tl, label) in enumerate([(cs_clim, tl_c, "climatology"), (cs_meas, tl_m, "measured")]):
        ax = axes[row_i, col]
        ax.plot(times, cs["ghi"], label="GHI")
        ax.plot(times, cs["dni"], label="DNI")
        ax.plot(times, cs["dhi"], label="DHI")
        ax.set_title(f"{period} — {label} T$_L$={tl:.2f}")
        ax.text(0.02, 0.97, f"max GHI: {cs['ghi'].max():.0f} W/m²\nmax DNI: {cs['dni'].max():.0f} W/m²",
                transform=ax.transAxes, fontsize=8, va="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
        if row_i == 0 and col == 0:
            ax.legend(loc="upper right", fontsize=8)
axes[0,0].set_ylabel("Irradiance (W/m²)")
axes[1,0].set_ylabel("Irradiance (W/m²)")
fig.suptitle("Clear-Sky Model Output: Climatology vs Self-Computed T$_L$\n(lowest and highest climatological-turbidity months)")
plt.tight_layout()
plt.show()


Using climatology's higher T_L understates achievable clear-sky irradiance relative to what
the station actually experiences — directly relevant for solar resource assessment and PV
yield prediction at this site.


## Sub-case: an extreme deviation, explained (June 18, 2022)

The single highest T_L values of the entire year (up to 6.89, roughly double the typical ~2.9)
all occurred on one day : **June 18, 2022, 11:00 UTC.**

In [ ]:
day = ds.sel(time="2022-06-18")
fig, ax = plt.subplots(figsize=(11,4.5))
ax.plot(day.time, day["DNI"], label="Measured DNI")
ax.plot(day.time, day["cDNI"], "--", label="Modelled clear-sky DNI")
ax.axvspan(pd.Timestamp("2022-06-18 06:00"), pd.Timestamp("2022-06-18 10:30"),
           color="red", alpha=0.15, label="Elevated T$_L$ window")
ax.set_title("June 18, 2022 — Measured vs Modelled DNI")
ax.set_ylabel("W/m²"); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
window = ds["TL_derived"].sel(time=slice("2022-06-18 03:00", "2022-06-18 13:00")).to_series()
half_hourly = window.resample("30min").mean()
half_hourly_clean = half_hourly.dropna()
print(half_hourly_clean)

Median turbidity value for June was 2.9

### Satellite confirmation (NASA Worldview, MODIS)

[MODIS AOD over Oldenburg, June 18 2022]

elevated AOD (~0.48) visible across the region surrounding Oldenburg. 

### MODIS visual context

The original working notebook referenced a MODIS Worldview screenshot stored on the author's computer. That image is not included in this repository. To reproduce this visual context, export the relevant image from NASA Worldview and place it in `figures/`, then embed it here using a relative path. The quantitative aerosol cross-check in this notebook is documented separately using CAMS reanalysis.

### Independent corroboration: aerosol optical depth (CAMS reanalysis)

Total AOD at 550 nm (Copernicus CAMS EAC4 reanalysis, nearest grid point to OLUOL) confirms a
multi-day elevated-aerosol event spanning June 17–19, well above the typical clean-air
background of ~0.1:


In [ ]:
aod_timeline = pd.Series(
    data=[0.1365, 0.1891, 0.2031, 0.2649, 0.5422, 0.4165, 0.2678, 0.2002,
          0.2073, 0.2140, 0.2549, 0.2639, 0.4954, 0.5952, 0.5673, 0.5118,
          0.3866, 0.3984, 0.4467, 0.5070, 0.4880, 0.4143, 0.2163, 0.1532],
    index=pd.date_range("2022-06-17", periods=24, freq="3h")
)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(aod_timeline.index, aod_timeline.values, marker="o", label="AOD")
ax.axvspan(pd.Timestamp("2022-06-18 08:00"), pd.Timestamp("2022-06-18 12:00"),
           color="red", alpha=0.2, label="Elevated T$_L$ window")
ax.axhline(0.1, color="gray", linestyle=":", label="typical clean-air AOD (~0.1)")
ax.set_ylabel("AOD (550 nm)")
ax.set_title("Aerosol Optical Depth over Oldenburg — CAMS Reanalysis, June 17–19, 2022")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Species breakdown** (CAMS, June 18, 11:00 UTC) confirms **mineral dust** as the dominant
contributor — roughly double the next-largest species (organic matter), and consistently
so throughout the day:

| Species | AOD @ 11:00 UTC | Share |
|---|---|---|
| **Dust** | 0.25 | ~51% |
| Organic matter | 0.12 | ~25% |
| Sulphate | 0.10 | ~18% |
| Sea salt | 0.02 | ~4% |
| Black carbon | 0.01 | ~2% |

*Data source: Copernicus Atmosphere Monitoring Service (CAMS), global reanalysis EAC4,
Copernicus Atmosphere Data Store.*







**Answer to research question 4:** yes - a clear, independently-corroborated extreme deviation was identified and attributed to a dust transport event, though no single named/documented event could be confirmed via general news search for this specific date.


# Conclusions and Future Work

### Conclusions
- Climatological Linke turbidity **systematically overestimates** the true clear-sky turbidity
  at OLUOL, by 15–41% depending on month, in every month studied
- This bias likely reflects both a genuine local/climatological mismatch **and** a selection
  effect from restricting self-computed T_L to only the cleanest clear-sky moments
- A specific extreme deviation (June 18, 2022) was identified, physically explained
  (DNI-suppressed / GHI-normal signature), and independently corroborated via satellite (MODIS)
  and reanalysis (CAMS) aerosol data, attributed primarily to mineral dust

### Limitations
- One year of station data vs. a **multi-year** climatological average — some of the gap may
  reflect this specific year being atypical
- Clear-sky sample sizes vary by month (as few as 185, as many as 3211 points)

### Future work
- Multi-year comparison to avoid the limitation of sample counts.
- Apply the same pipeline to other Eye2Sky stations for a regional picture
- Automated dust/aerosol-event detection across the full year, not just the top outlier
